In [9]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy.stats import t
from pathlib import Path
from joblib import Parallel, delayed
import json

In [10]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'mining-area').mkdir(parents=True, exist_ok=True)

# Estimate mining-area expansion (2050) — per CE scenario × β

Converts each allocated-extraction raster (50 km, Mt/cell, from `5-2-map-extraction`) into the **mining footprint area** (km²/cell) it implies, by distributing each cell's extraction across its 1 km sub-cells (ranked by suitability × reserve) and applying the empirical extraction→area power law `A(ha) = a·E(Mt)^b` (fitted in `3-fit-extraction-area-relationship`).

## Inputs (all on the model `results/`)
- allocated extraction: `extraction-rasters/allocated-extraction-2023-2050-{scenario}-B{beta}-{res}-D{Ld}-R{Lr}.tif` (no area dimension — area does not enter the allocation)
- 1 km capacity (Mt/cell): `mining-area/aggregate-reserve-1km-{Lr}.tif`
- 1 km suitability: `global-mining-suitability-1km.tif`
- area–extraction data: `extraction-area-relationship.csv`

## Cross (central case)
scenario × β sweep at the central demand/reserve/area case → `mining-area/mining-area-expansion-2050-{scenario}-B{beta}-{res}-D{Ld}-R{Lr}-A{La}.tif`.

The output carries all three uncertainty dimensions: `-DM-RM-AM` (central), `-DL-RH-AL` (lower bound), `-DH-RL-AH` (upper bound).

In [3]:
# shared 1 km layers (grid + suitability); reused across all scenario × beta runs
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
country_raster_50km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif')
mining_suitability_1km = rio.open(RESULTS / 'global-mining-suitability-1km.tif').read(1)

In [4]:
def func_1(x, a, b):
    return a * x ** b

In [5]:
# empirical extraction (Mt) -> disturbed area (ha) power law (same data as notebook 3)
df_rel = pd.read_csv(RESULTS / 'extraction-area-relationship.csv')
X_rel = df_rel['E (Mt)'].values
Y_rel = df_rel['A (ha)'].values
popt_1, pcov_1 = curve_fit(func_1, X_rel, Y_rel)
a_1, b_1 = popt_1
dof_1 = len(X_rel) - len(popt_1)
print(f'A = {a_1:.3f} * E^{b_1:.3f}  (n={len(X_rel)})')

A = 9.421 * E^0.548  (n=113)


In [6]:
def predict_interval(X, popt, pcov, dof):
    """95% prediction band for the power law (used when area_level in {'L','H'})."""
    a, b = popt
    cov_aa, cov_ab, cov_bb = pcov[0, 0], pcov[0, 1], pcov[1, 1]
    mask_valid = (X > 0)
    X_safe = np.where(mask_valid, X, 1e-12)
    y_best = a * X_safe ** b
    dy_da = X_safe ** b
    dy_db = a * X_safe ** b * np.log(X_safe)
    sigma_y2 = cov_aa * dy_da ** 2 + 2 * cov_ab * dy_da * dy_db + cov_bb * dy_db ** 2
    sigma_y = np.sqrt(sigma_y2)
    tval = t.ppf(0.975, dof)
    lower = np.where(mask_valid, y_best - tval * sigma_y, 0)
    upper = np.where(mask_valid, y_best + tval * sigma_y, 0)
    return lower.astype(np.float32), upper.astype(np.float32)

In [7]:
def estimate_mining_area(scenario, beta, demand_level='M', reserve_level='M', area_level='M', res='50km',
                         suit_1km=None, cap_1km=None, fit=None):
    """Distribute each 50 km cell's allocated extraction to 1 km sub-cells (ranked by
    suitability x reserve) and sum the power-law footprint area. area_level in {'L','M','H'}.

    NUMERICALLY IDENTICAL to the original; two parallel-/speed-oriented changes only:
      * the 50km->1km index offset is CONSTANT, so it is computed once instead of re-opening two
        rasters inside the per-cell loop (was ~2 x 25k rio.open per combo);
      * the shared read-only 1 km layers + power-law fit can be passed in (loaded once) so joblib
        memmaps them across workers instead of each task re-reading / re-pickling them:
            suit_1km : 1 km mining-suitability array        (else read from disk)
            cap_1km  : 1 km aggregate-reserve, Mt/cell       (else read from disk, reserve_level)
            fit      : (a, b, popt, pcov, dof)               (else module globals a_1, b_1, ...)

    File tags: the allocated-extraction INPUT depends only on demand x reserve (`alloc_tag`, area
    does not enter the allocation), but the footprint OUTPUT also carries the area level
    (`out_tag = alloc_tag + '-A{area_level}'`), so all three uncertainty dimensions are explicit
    in the filename: -DM-RM-AM (central), -DL-RH-AL (lower), -DH-RL-AH (upper).
    Returns a one-line summary string; the footprint raster is written to disk."""
    if suit_1km is None:
        suit_1km = rio.open(RESULTS / 'global-mining-suitability-1km.tif').read(1)
    if cap_1km is None:
        cap_1km = rio.open(RESULTS / 'mining-area' / f'aggregate-reserve-1km-{reserve_level}.tif').read(1)
    a, b, popt, pcov, dof = fit if fit is not None else (a_1, b_1, popt_1, pcov_1, dof_1)

    # 50 km -> 1 km mapping: scale + origin offset are constants -> compute ONCE (not per cell)
    src_ref = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    dst_ref = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
    scale = {'10km': 10, '50km': 50}[res]
    off_i = int(abs(dst_ref.transform[5] - src_ref.transform[5]) / 1e3)
    off_j = int(abs(dst_ref.transform[2] - src_ref.transform[2]) / 1e3)

    alloc_tag = f'{scenario}-B{beta:.2f}-{res}-D{demand_level}-R{reserve_level}'   # allocation: no area dim
    out_tag   = f'{alloc_tag}-A{area_level}'                                       # footprint: + area dim
    allocated_extraction = rio.open(
        RESULTS / 'extraction-rasters' / f'allocated-extraction-2023-2050-{alloc_tag}.tif').read(1)

    mining_area_expansion = np.zeros(src_ref.shape, dtype=np.float32)
    i_set, j_set = np.where(allocated_extraction > 0)
    for i, j in zip(i_set, j_set):
        i_1, i_2 = i * scale - off_i, (i + 1) * scale - off_i
        j_1, j_2 = j * scale - off_j, (j + 1) * scale - off_j
        suitability_slice = suit_1km[i_1:i_2, j_1:j_2] / 1e4
        avail_slice = cap_1km[i_1:i_2, j_1:j_2]
        Q_total = allocated_extraction[i, j]

        score_flat = (suitability_slice * avail_slice).flatten()
        avail_flat = avail_slice.flatten()
        m = avail_flat > 0
        score_flat, avail_flat = score_flat[m], avail_flat[m]
        if score_flat.size == 0:
            continue

        order = np.argsort(score_flat)[::-1]            # best (most suitable, highest reserve) first
        q_max = avail_flat[order]
        q_assigned = np.zeros_like(q_max)
        Q_remain = Q_total
        for k in range(len(q_max)):                     # greedy fill — kept EXACTLY (float32 order matters)
            q_k = min(Q_remain, q_max[k])
            q_assigned[k] = q_k
            Q_remain -= q_k
            if Q_remain <= 0:
                break

        if area_level == 'M':
            area_each = (a * q_assigned ** b) / 1e2                          # func_1(.)/1e2; ha -> km^2
        else:
            lo, hi = predict_interval(q_assigned, popt, pcov, dof)
            area_each = (lo if area_level == 'L' else hi) / 1e2
        mining_area_expansion[i, j] = float(np.sum(area_each))

    total_expansion  = float(np.sum(mining_area_expansion))
    total_extraction = float(np.sum(allocated_extraction)) / 1e3

    out_path = RESULTS / 'mining-area' / f'mining-area-expansion-2050-{out_tag}.tif'
    with rio.open(out_path, 'w', driver='GTiff',
        height=src_ref.shape[0], width=src_ref.shape[1], count=1,
        dtype=mining_area_expansion.dtype, crs=src_ref.crs,
        transform=src_ref.transform, compress='lzw') as dst:
        dst.write(mining_area_expansion, 1)
    return f'{out_tag}: extraction {total_extraction:.1f} Gt -> footprint {total_expansion:,.0f} km^2'

## Run — `scenario_set_1` × β0 × full 3×3×3 uncertainty factorial (216 footprints)

Every scenario in `scenario_set_1` at β = 0, swept over the **full L/M/H factorial of all three uncertainty sources** (demand × reserve × area) = 8 × 3 × 3 × 3 = **216** footprint rasters.

Unlike the 4-signpost anti-diagonal bounds cell below, this enumerates *every* corner of the D×R×A cube, so `cap_1km` is matched to each task's reserve level (RH→`-1km-H`, RM→`-1km-M`, RL→`-1km-L`).

→ `mining-area/mining-area-expansion-2050-{scen}-B0.00-50km-D{Ld}-R{Lr}-A{La}.tif`

In [11]:
scenario_set_1 = ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']
LEVELS = ['L', 'M', 'H']
N_JOBS = 10                                          # processes; raise toward os.cpu_count()-2 if RAM allows

# Self-contained shared inputs (does NOT depend on the cross / bounds cells below):
#   suitability + power-law fit come from the early cells; cap_1km MUST match the reserve level,
#   so preload all three reserve rasters once and pick per-task (joblib memmaps them to workers).
suit_1km       = mining_suitability_1km
fit            = (a_1, b_1, popt_1, pcov_1, dof_1)
cap_by_reserve = {lvl: rio.open(RESULTS / 'mining-area' / f'aggregate-reserve-1km-{lvl}.tif').read(1)
                  for lvl in LEVELS}

# Full factorial: 8 scenarios x {D,R,A} in {L,M,H}^3 = 8 x 27 = 216 footprints, all at beta = 0.
combos = [(scen, dl, rl, al)
          for scen in scenario_set_1
          for dl in LEVELS for rl in LEVELS for al in LEVELS]

msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(scen, 0.0, demand_level=dl, reserve_level=rl, area_level=al,
                                  suit_1km=suit_1km, cap_1km=cap_by_reserve[rl], fit=fit)
    for scen, dl, rl, al in combos
)
for msg in msgs:
    print(' ', msg)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:    7.0s
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:    9.9s
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed:   13.7s
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:   16.7s
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed:   21.4s
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed:   25.3s
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed:   30.4s
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed:   35.2s
[Parallel(n_jobs=10)]: Done  93 tasks      | elapsed:   41.5s
[Parallel(n_jobs=10)]: Done 108 tasks      | elapsed:   47.0s
[Parallel(n_jobs=10)]: Done 125 tasks      | elapsed:   53.9s
[Parallel(n_jobs=10)]: Done 142 tasks      | elapsed:   59.5s
[Parallel(n_jobs=10)]: Done 161 tasks      | elapsed:  1.1min
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:  1.2min
[Parallel(n_jobs=10)]: Done 216 out of 216 | elapsed:  

  BAU-B0.00-50km-DL-RL-AL: extraction 378.7 Gt -> footprint 9,560 km^2
  BAU-B0.00-50km-DL-RL-AM: extraction 378.7 Gt -> footprint 11,147 km^2
  BAU-B0.00-50km-DL-RL-AH: extraction 378.7 Gt -> footprint 12,735 km^2
  BAU-B0.00-50km-DL-RM-AL: extraction 378.7 Gt -> footprint 7,794 km^2
  BAU-B0.00-50km-DL-RM-AM: extraction 378.7 Gt -> footprint 8,934 km^2
  BAU-B0.00-50km-DL-RM-AH: extraction 378.7 Gt -> footprint 10,073 km^2
  BAU-B0.00-50km-DL-RH-AL: extraction 378.7 Gt -> footprint 7,077 km^2
  BAU-B0.00-50km-DL-RH-AM: extraction 378.7 Gt -> footprint 8,100 km^2
  BAU-B0.00-50km-DL-RH-AH: extraction 378.7 Gt -> footprint 9,123 km^2
  BAU-B0.00-50km-DM-RL-AL: extraction 445.2 Gt -> footprint 11,002 km^2
  BAU-B0.00-50km-DM-RL-AM: extraction 445.2 Gt -> footprint 12,761 km^2
  BAU-B0.00-50km-DM-RL-AH: extraction 445.2 Gt -> footprint 14,520 km^2
  BAU-B0.00-50km-DM-RM-AL: extraction 445.2 Gt -> footprint 8,900 km^2
  BAU-B0.00-50km-DM-RM-AM: extraction 445.2 Gt -> footprint 10,143 km^2

## Run — scenario × β cross (30 footprints, M/M)

In [ ]:
_cfg = json.load(open(RESULTS / 'sweep-config.json'))   # sweep grid defined once in 5-2-map-extraction
SCENARIOS, BETAS = _cfg['SCENARIOS'], _cfg['BETAS']
N_JOBS    = 10                                                   # processes; raise toward (os.cpu_count()-2) if RAM allows

# Load the shared read-only 1 km layers ONCE. Passing them as args lets joblib memmap them to the
# workers (dumped once, mmap'd read-only) instead of each of the 50-60 tasks re-reading from disk
# or each pickling a full copy -> low memory, no per-task I/O.
suit_1km = mining_suitability_1km                              # already loaded in the shared-layers cell
cap_1km  = rio.open(RESULTS / 'mining-area' / 'aggregate-reserve-1km-M.tif').read(1)   # reserve_level='M'
fit      = (a_1, b_1, popt_1, pcov_1, dof_1)

combos = [(scen, b) for scen in SCENARIOS for b in BETAS]
msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(scen, b, demand_level='M', reserve_level='M', area_level='M',
                                  suit_1km=suit_1km, cap_1km=cap_1km, fit=fit)
    for scen, b in combos
)
for msg in msgs:
    print(' ', msg)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:    3.1s
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:    3.3s
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed:    4.4s
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    5.4s
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed:    6.5s
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed:    7.0s
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed:    8.2s
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed:    9.4s
[Parallel(n_jobs=10)]: Done  93 tasks      | elapsed:   11.2s
[Parallel(n_jobs=10)]: Done 108 tasks      | elapsed:   12.4s
[Parallel(n_jobs=10)]: Done 139 out of 143 | elapsed:   14.8s remaining:    0.3s
[Parallel(n_jobs=10)]: Done 143 out of 143 | elapsed:   15.2s finished


  BAU-B0.00-50km-DM-RM: extraction 445.2 Gt -> footprint 10,143 km^2
  BAU-B0.10-50km-DM-RM: extraction 445.2 Gt -> footprint 10,148 km^2
  BAU-B0.20-50km-DM-RM: extraction 445.2 Gt -> footprint 10,128 km^2
  BAU-B0.30-50km-DM-RM: extraction 445.2 Gt -> footprint 10,036 km^2
  BAU-B0.40-50km-DM-RM: extraction 445.2 Gt -> footprint 9,905 km^2
  BAU-B0.50-50km-DM-RM: extraction 445.2 Gt -> footprint 9,715 km^2
  BAU-B0.60-50km-DM-RM: extraction 445.2 Gt -> footprint 9,476 km^2
  BAU-B0.70-50km-DM-RM: extraction 445.2 Gt -> footprint 9,213 km^2
  BAU-B0.80-50km-DM-RM: extraction 445.2 Gt -> footprint 8,959 km^2
  BAU-B0.85-50km-DM-RM: extraction 445.2 Gt -> footprint 8,789 km^2
  BAU-B0.90-50km-DM-RM: extraction 445.2 Gt -> footprint 8,629 km^2
  BAU-B0.95-50km-DM-RM: extraction 445.2 Gt -> footprint 8,496 km^2
  BAU-B1.00-50km-DM-RM: extraction 445.2 Gt -> footprint 8,377 km^2
  CE10-B0.00-50km-DM-RM: extraction 433.8 Gt -> footprint 9,932 km^2
  CE10-B0.10-50km-DM-RM: extraction 433.8 G

## Uncertainty bounds — anti-diagonal footprints at the 4 signposts

The three uncertainty sources do not all move the footprint the same way: demand and the area-CI band increase it, but **reserve decreases it** (more reserve per cell concentrates extraction into fewer sub-cells → less area under the concave power law). So the true extremes are the anti-diagonal corners:

- **lower = `DL-RH` + `area_level='L'`** → `…-DL-RH-AL.tif`
- **upper = `DH-RL` + `area_level='H'`** → `…-DH-RL-AH.tif`

`cap_1km` is matched to the reserve level (`RH`→1km-H, `RL`→1km-L). The footprint filename now carries the area level explicitly (`-A{La}`), so all three dimensions are visible; the allocated-extraction input it reads stays `D{}-R{}` (area does not affect allocation).

In [ ]:
# --- Footprint bounds: the two ANTI-DIAGONAL corners that minimize / maximize the outcome.
#     Reserve runs OPPOSITE to demand (low reserve spreads extraction -> larger footprint via the
#     concave area law), so the extremes are NOT the all-low/all-high diagonal but:
#         lower bound = DL-RH + area L  (low demand, high reserve/concentrate, low area-CI edge)
#         upper bound = DH-RL + area H  (high demand, low reserve/spread,      high area-CI edge)
#     cap_1km must match the reserve level: RH -> aggregate-reserve-1km-H, RL -> -1km-L.
#     The central case (DM-RM, area M) already exists above (part of the 143-combo sweep).
#       -> mining-area-expansion-2050-{scen}-B{b}-50km-{DL-RH | DH-RL}.tif
cap_1km_L = rio.open(RESULTS / 'mining-area' / 'aggregate-reserve-1km-L.tif').read(1)
cap_1km_H = rio.open(RESULTS / 'mining-area' / 'aggregate-reserve-1km-H.tif').read(1)

SIGNPOSTS    = [('BAU', 0.0), ('BAU', 0.6), ('CE', 0.0), ('CE', 0.6)]   # BAU / CON / CE / IAP
bound_combos = ([(s, b, 'L', 'H', 'L', cap_1km_H) for s, b in SIGNPOSTS]   # lower = DL-RH, area L
              + [(s, b, 'H', 'L', 'H', cap_1km_L) for s, b in SIGNPOSTS])  # upper = DH-RL, area H

bound_msgs = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
    delayed(estimate_mining_area)(s, b, demand_level=dl, reserve_level=rl, area_level=al,
                                  suit_1km=suit_1km, cap_1km=cap, fit=fit)
    for (s, b, dl, rl, al, cap) in bound_combos
)
for m in bound_msgs:
    print(' ', m)